In [1]:
# ===== USER PARAMETERS =====
TABLE_NAME = "Demo"        # your Lakehouse table name
ID_COL     = "flight"      # or "hex", "icao", etc.
TS_COL     = "dt"          # timestamp column (Spark Timestamp or epoch seconds)
LAT_COL    = "lat"
LON_COL    = "lon"
ALT_COL    = "altitude"    # optional (can be None)
SPD_COL    = "speed" # optional (can be None)

# Narrow to a region + timeframe for performance
START_TS = "2020-01-01 00:00:00"   # Start and end times to limit scope to a specific time frame
END_TS   = "2020-12-30 00:00:00"

# Bounding box (example: Minneapolis area — adjust as needed)
MIN_LAT, MAX_LAT = 32.6317, 32.9217    # Define a bounding box to limit search to a specific geographic area.  In this case the bounding box
MIN_LON, MAX_LON = -96.9690, -96.6250  # is 5 square miles directly over Downtown Dallas, Texas

# Circle/loiter thresholds
WINDOW_MINUTES = 10          # window length
STEP_MINUTES   = 5           # slide step
MIN_POINTS     = 30          # minimum points in a window
MIN_LOOPS      = 1.25        # loops threshold (1.0 = full circle)
MAX_SPEED_KTS  = 300         # Can be used to filter slower traffic (helicopters/law enforcement may be slower)
MIN_SPEED_KTS  = 30
MIN_ALT_FT     = 0           # Can adjust to look in a specific altitude range
MAX_ALT_FT     = 30000



StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df = spark.read.table(TABLE_NAME)

# Handle timestamp types (epoch seconds -> timestamp) if needed
# If your TS_COL is already timestamp, this will be a no-op-ish
df = df.withColumn("ts", F.col(TS_COL).cast("timestamp"))

# Basic spatial + temporal filter
df_f = (
    df
    .filter((F.col("ts") >= F.to_timestamp(F.lit(START_TS))) & (F.col("ts") < F.to_timestamp(F.lit(END_TS))))
    .filter((F.col(LAT_COL) >= MIN_LAT) & (F.col(LAT_COL) <= MAX_LAT) &
            (F.col(LON_COL) >= MIN_LON) & (F.col(LON_COL) <= MAX_LON))
    .select(
        F.col(ID_COL).alias("aircraft_id"),
        F.col("ts"),
        F.col(LAT_COL).cast("double").alias("lat"),
        F.col(LON_COL).cast("double").alias("lon"),
        (F.col(ALT_COL).cast("double").alias("alt_ft") if ALT_COL else F.lit(None).cast("double").alias("alt_ft")),
        (F.col(SPD_COL).cast("double").alias("spd_kts") if SPD_COL else F.lit(None).cast("double").alias("spd_kts")),
    )
    .dropna(subset=["aircraft_id","ts","lat","lon"])
)

# Optional sanity filters
if ALT_COL:
    df_f = df_f.filter((F.col("alt_ft") >= MIN_ALT_FT) & (F.col("alt_ft") <= MAX_ALT_FT))
if SPD_COL:
    df_f = df_f.filter((F.col("spd_kts") >= MIN_SPEED_KTS) & (F.col("spd_kts") <= MAX_SPEED_KTS))

df_f.cache()
display(df_f.limit(10))

StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7bde1614-06f2-4f4f-b1ec-e99ca45490e3)

In [3]:
import math
from pyspark.sql.types import DoubleType

# Great-circle distance in meters
@F.udf(DoubleType())
def haversine_m(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6371000.0
    p1 = math.radians(lat1); p2 = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlon/2)**2
    return 2*R*math.asin(math.sqrt(a))

# Bearing in radians (-pi..pi)
@F.udf(DoubleType())
def bearing_rad(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    p1 = math.radians(lat1); p2 = math.radians(lat2)
    dlon = math.radians(lon2 - lon1)
    y = math.sin(dlon) * math.cos(p2)
    x = math.cos(p1)*math.sin(p2) - math.sin(p1)*math.cos(p2)*math.cos(dlon)
    return math.atan2(y, x)

# Wrap angle difference to (-pi..pi)
@F.udf(DoubleType())
def wrap_angle(d):
    if d is None:
        return None
    while d <= -math.pi:
        d += 2*math.pi
    while d > math.pi:
        d -= 2*math.pi
    return d


StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 5, Finished, Available, Finished, False)

In [4]:
# Add sliding window
df_w = (
    df_f
    .withColumn("win", F.window("ts", f"{WINDOW_MINUTES} minutes", f"{STEP_MINUTES} minutes"))
    .select(
        "aircraft_id", "ts", "lat", "lon", "alt_ft", "spd_kts",
        F.col("win.start").alias("win_start"),
        F.col("win.end").alias("win_end")
    )
)

# Order within each aircraft_id + window
w = Window.partitionBy("aircraft_id","win_start","win_end").orderBy("ts")

df_seq = (
    df_w
    .withColumn("lat_prev", F.lag("lat").over(w))
    .withColumn("lon_prev", F.lag("lon").over(w))
    .withColumn("ts_prev",  F.lag("ts").over(w))
    .withColumn("bearing", bearing_rad(F.col("lat_prev"),F.col("lon_prev"),F.col("lat"),F.col("lon")))
    .withColumn("bearing_prev", F.lag("bearing").over(w))
    .withColumn("dtheta", wrap_angle(F.col("bearing") - F.col("bearing_prev")))
    .withColumn("seg_m", haversine_m(F.col("lat_prev"),F.col("lon_prev"),F.col("lat"),F.col("lon")))
)

# Aggregate per window
agg = (
    df_seq
    .groupBy("aircraft_id","win_start","win_end")
    .agg(
        F.count("*").alias("n_points"),
        F.sum(F.when(F.col("dtheta").isNotNull(), F.abs(F.col("dtheta"))).otherwise(F.lit(0.0))).alias("sum_abs_turn"),
        F.sum(F.when(F.col("dtheta").isNotNull(), F.col("dtheta")).otherwise(F.lit(0.0))).alias("sum_signed_turn"),
        F.sum(F.when(F.col("seg_m").isNotNull(), F.col("seg_m")).otherwise(F.lit(0.0))).alias("path_m"),
        F.avg("spd_kts").alias("avg_spd_kts"),
        F.avg("alt_ft").alias("avg_alt_ft"),
        F.min("ts").alias("min_ts"),
        F.max("ts").alias("max_ts"),
    )
    .filter(F.col("n_points") >= MIN_POINTS)
)

# Estimate loops: signed turn / (2*pi)
agg = agg.withColumn("loops_signed", F.col("sum_signed_turn") / F.lit(2*math.pi)) \
         .withColumn("loops_abs",    F.col("sum_abs_turn")    / F.lit(2*math.pi))

StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 6, Finished, Available, Finished, False)

In [5]:
# Centroid per window
cent = (
    df_w.groupBy("aircraft_id","win_start","win_end")
    .agg(F.avg("lat").alias("clat"), F.avg("lon").alias("clon"))
)

df_r = (
    df_w.join(cent, ["aircraft_id","win_start","win_end"], "inner")
        .withColumn("r_m", haversine_m(F.col("clat"),F.col("clon"),F.col("lat"),F.col("lon")))
)

rstats = (
    df_r.groupBy("aircraft_id","win_start","win_end")
        .agg(
            F.avg("r_m").alias("avg_r_m"),
            F.stddev("r_m").alias("std_r_m"),
            F.expr("percentile_approx(r_m, 0.9)").alias("p90_r_m"),
        )
)

scored = (
    agg.join(rstats, ["aircraft_id","win_start","win_end"], "left")
       .withColumn("radius_cv", F.col("std_r_m") / F.col("avg_r_m"))
)


StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 7, Finished, Available, Finished, False)

In [6]:
candidates = (
    scored
    # Prefer consistent turning direction (signed loops)
    .filter(F.abs(F.col("loops_signed")) >= MIN_LOOPS)
    # Radius stability guardrail (tune as needed)
    .filter((F.col("avg_r_m") >= 300) & (F.col("avg_r_m") <= 50000))  # ignore tiny jitter / huge arcs
    .filter((F.col("radius_cv").isNull()) | (F.col("radius_cv") <= 0.35))
    .withColumn("turn_direction", F.when(F.col("loops_signed") > 0, F.lit("CCW")).otherwise(F.lit("CW")))
    .withColumn("loops", F.abs(F.col("loops_signed")))
    .orderBy(F.desc("loops"), F.asc("radius_cv"))
)

display(candidates.limit(50))

StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10a3b884-3fe5-4dce-b461-85a60c35bcdf)

In [7]:
OUT_TABLE = "Demo_CircleCandidates"

(
    candidates
    .select(
        "aircraft_id","win_start","win_end","loops","turn_direction",
        "n_points","path_m","avg_spd_kts","avg_alt_ft",
        "avg_r_m","p90_r_m","radius_cv"
    )
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(OUT_TABLE)
)

print(f"Wrote {OUT_TABLE}")


StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 9, Finished, Available, Finished, False)

Wrote Demo_CircleCandidates


In [8]:
# Pick top result
top = candidates.select("aircraft_id","win_start","win_end").limit(1).collect()
if top:
    aid = top[0]["aircraft_id"]
    ws  = top[0]["win_start"]
    we  = top[0]["win_end"]

    pts = (
        df_f.filter(F.col("aircraft_id")==aid)
            .filter((F.col("ts") >= F.lit(ws)) & (F.col("ts") < F.lit(we)))
            .orderBy("ts")
    )
    display(pts)
else:
    print("No candidates found with current thresholds/filters.")

StatementMeta(, c3489c5d-f08b-49c3-8189-be1c14577160, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d2a4e719-8796-465f-a059-0216a435e6a1)